#RQ1: Role-Salary Prediction
**Dissertation:** AI Approaches to Analysing Recruitment Demand: Machine learning insights from European Pharmaceutical Job Postings  
**Author:** Kashmira Bhoir  
**Institution:** Gisma University of Applied Sciences  
**Date:** 27 March 2026  

## Research Question
To what extent can multi-task transformers accurately predict
pharmaceutical job roles and associated salary levels from job posting text?

## Overview
This notebook helps to build a dual-head multi-task transformer that together:
- Head 1: Classifies pharma job roles
- Head 2: Predicts associated salaries by level (€)

## Key Findings
- Role Classification Accuracy : 80.3%
- Overall Salary MAE           : €8,224
- Best per-role MAE            : €5,364 (Pharmaceutical, Healthcare And Medical Sales)

## Input
- `Combined_Pharma_Jobs_Cleaned.csv` — 8,826 rows, 13 columns

## Outcome
- Trained dual-head Keras model
- Role classification report (precision,F1, recall)
- Salary prediction for each role
- 4-run improvement tracker

## Research Gap Addressed
Ather et al. (2024) and Agrawal et al. (2025) focused exclusively
on US and UK technical roles. This is the first multi-task transformer for
joint pharmaceutical role + salary prediction on European job posting data.

Install and Import Libraries

In [ ]:
!pip install sentence-transformers tensorflow scikit-learn pandas numpy -q

import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, mean_absolute_error,
                             classification_report)
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

print(f" Libraries loaded")
print(f"   TensorFlow version : {tf.__version__}")
print(f"   Pandas version     : {pd.__version__}")

 Libraries loaded
   TensorFlow version : 2.19.0
   Pandas version     : 2.2.2


In [ ]:
file_path = "/content/Combined_Pharma_Jobs_Cleaned.csv"

df = pd.read_csv(file_path, on_bad_lines='skip', engine='python')
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print(f"Loaded: {df.shape[0]:,} rows  |  {df.shape[1]} columns")
print(f"\n   Columns: {df.columns.tolist()}")

Loaded: 9,420 rows  |  13 columns

   Columns: ['category', 'company_name', 'job_description', 'job_title', 'job_type', 'location', 'post_date', 'salary_offered', 'salary_num', 'job_type_clean', 'country', 'salary_min', 'salary_max']


In [ ]:
def parse_salary(val):
    if pd.isna(val): return None
    val_low = str(val).strip().lower()
    vague   = ['competitive','negotiable','neg ','on application',
               'tbc','doe','excellent','attractive','discussed',
               'depending on','uncapped','market rate']
    if any(v in val_low for v in vague): return None
    if not re.search(r'\d', val_low):    return None
    if re.search(r'p\s*hour|per\s*hour|/hr', val_low): return None
    s = re.sub(r'swiss\s*franc|chf|czk|us\$|gbp|eur|€|£|\$', '', val_low)
    s = re.sub(r'up\s*to|approx|from|starting\s*at', '', s)
    s = re.sub(r'(pa|per\s*annum|p\.a\.|pm|per\s*month).*$', '', s)
    s = re.sub(r'(\d),(\d)', r'\1\2', s)
    s = re.sub(r'\+.*$', '', s).strip()
    has_k   = bool(re.search(r'\d\s*k', val_low))
    numbers = re.findall(r'\d+\.?\d*', s)
    if not numbers: return None
    nums = [float(n) * (1000 if has_k and float(n) < 1000 else 1)
            for n in numbers]
    nums = [n for n in nums if 10000 <= n <= 500000]
    if not nums: return None
    return (nums[0] + nums[1]) / 2 if len(nums) >= 2 else nums[0]

df['salary_num'] = df['salary_offered'].apply(parse_salary)

parseable = df['salary_num'].notna().sum()
print(f"Salary parsed")
print(f"   Parseable : {parseable:,}  ({parseable/len(df)*100:.1f}%)")
print(f"   Hidden    : {len(df)-parseable:,}  ({(len(df)-parseable)/len(df)*100:.1f}%)")
print(f"\n   Salary stats (€):")
print(df['salary_num'].describe().apply(lambda x: f"   {x:,.0f}").to_string())

Salary parsed
   Parseable : 1,444  (15.3%)
   Hidden    : 7,976  (84.7%)

   Salary stats (€):
count         1,444
mean         56,620
std          28,938
min          10,000
25%          37,500
50%          50,000
75%          62,500
max         230,000


In [ ]:
KEEP_ROLES = [
    'Pharmaceutical, Healthcare And Medical Sales',
    'Clinical Research',
    'Manufacturing & Operations',
    'Pharmaceutical Marketing',
    'Regulatory Affairs',
]

df['category']   = df['category'].astype(str).str.strip().str.title()
df['role_label'] = df['category'].apply(
    lambda x: x if x in KEEP_ROLES else None
)

df_joint = df[
    df['role_label'].notna() &
    df['salary_num'].notna()
].copy()

print(f"Role categories (top 5 — Science + QA removed):")
rc = df_joint['role_label'].value_counts().reset_index()
rc.columns = ['Role', 'Count']
rc['Share %'] = (rc['Count'] / len(df_joint) * 100).round(1)
print(rc.to_string(index=False))
print(f"\n   Total joint rows: {len(df_joint):,}")

Role categories (top 5 — Science + QA removed):
                                        Role  Count  Share %
Pharmaceutical, Healthcare And Medical Sales    511     46.9
                  Manufacturing & Operations    168     15.4
                           Clinical Research    166     15.2
                    Pharmaceutical Marketing    127     11.7
                          Regulatory Affairs    118     10.8

   Total joint rows: 1,090


In [ ]:
def build_text(row):
    return (f"{row.get('job_title','')}. "
            f"{row.get('location','')}. "
            f"{row.get('job_type','')}. "
            f"{str(row.get('job_description',''))[:300]}")

df_joint['text_input'] = df_joint.apply(build_text, axis=1)

print(f"Text input built for {len(df_joint):,} rows")
print(f"\n   Sample text input:")
print(f"   {df_joint['text_input'].iloc[0][:200]}...")

Text input built for 1,090 rows

   Sample text input:
   Pharmacovigilance Expert. Paris. Full-Time. Cleanroom process validation....


In [ ]:
print(f" Loading sentence-transformer model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print(f" Generating embeddings for {len(df_joint):,} rows...")
embeddings = embedder.encode(
    df_joint['text_input'].tolist(),
    batch_size        = 64,
    show_progress_bar = True
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"   Each job posting → {embeddings.shape[1]}-dimensional vector")

 Loading sentence-transformer model...


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Generating embeddings for 1,090 rows...



Embeddings shape: (1090, 384)
   Each job posting → 384-dimensional vector


In [ ]:
le_role   = LabelEncoder()
y_role    = le_role.fit_transform(df_joint['role_label'])
n_classes = len(le_role.classes_)

y_salary   = df_joint['salary_num'].values
sal_mean   = y_salary.mean()
sal_std    = y_salary.std()
y_sal_norm = (y_salary - sal_mean) / sal_std

print(f"Labels encoded")
print(f"   Role classes ({n_classes}): {list(le_role.classes_)}")
print(f"\n   Salary normalisation:")
print(f"   Mean : €{sal_mean:,.0f}")
print(f"   Std  : €{sal_std:,.0f}")
print(f"   Range: {y_sal_norm.min():.2f} → {y_sal_norm.max():.2f}")

Labels encoded
   Role classes (5): ['Clinical Research', 'Manufacturing & Operations', 'Pharmaceutical Marketing', 'Pharmaceutical, Healthcare And Medical Sales', 'Regulatory Affairs']

   Salary normalisation:
   Mean : €53,654
   Std  : €26,563
   Range: -1.64 → 6.64


In [ ]:
X_train, X_test, \
yr_train, yr_test, \
ys_train, ys_test = train_test_split(
    embeddings, y_role, y_sal_norm,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y_role
)

print(f"Train/Test split:")
print(f"   Train rows : {len(X_train):,}")
print(f"   Test rows  : {len(X_test):,}")
print(f"\n   Class distribution (train):")
for i, cls in enumerate(le_role.classes_):
    count = (yr_train == i).sum()
    print(f"   {cls:<45} : {count:,}")

Train/Test split:
   Train rows : 872
   Test rows  : 218

   Class distribution (train):
   Clinical Research                             : 133
   Manufacturing & Operations                    : 134
   Pharmaceutical Marketing                      : 102
   Pharmaceutical, Healthcare And Medical Sales  : 409
   Regulatory Affairs                            : 94


In [ ]:
inputs = keras.Input(shape=(384,), name='embedding_input')

shared = layers.Dense(256, activation='relu')(inputs)
shared = layers.BatchNormalization()(shared)
shared = layers.Dropout(0.3)(shared)
shared = layers.Dense(128, activation='relu')(shared)
shared = layers.BatchNormalization()(shared)
shared = layers.Dropout(0.3)(shared)

role_h   = layers.Dense(64, activation='relu')(shared)
role_h   = layers.Dropout(0.3)(role_h)
role_out = layers.Dense(
    n_classes, activation='softmax', name='role_output'
)(role_h)

sal_h   = layers.Dense(64, activation='relu')(shared)
sal_h   = layers.Dense(32, activation='relu')(sal_h)
sal_out = layers.Dense(
    1, activation='linear', name='salary_output'
)(sal_h)

model = keras.Model(inputs=inputs, outputs=[role_out, sal_out])

model.compile(
    optimizer    = keras.optimizers.Adam(learning_rate=3e-4),
    loss         = {
        'role_output'  : 'sparse_categorical_crossentropy',
        'salary_output': 'mse'
    },
    loss_weights = {'role_output': 1.0, 'salary_output': 1.0},
    metrics      = {
        'role_output'  : 'accuracy',
        'salary_output': 'mae'
    }
)

print(f"Model built — {model.count_params():,} total parameters")
model.summary()

Model built — 151,942 total parameters


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ embedding_input     │ (None, 384)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 256)       │     98,560 │ embedding_input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_5[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │     32,896 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_6[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 64)        │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 32)        │      2,080 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ role_output (Dense) │ (None, 5)         │        325 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ salary_output       │ (None, 1)         │         33 │ dense_9[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 151,942 (593.52 KB)

 Trainable params: 151,174 (590.52 KB)

 Non-trainable params: 768 (3.00 KB)

In [ ]:
print(f"  TRAINING — MULTITASK TRANSFORMER")

history = model.fit(
    X_train,
    {'role_output': yr_train, 'salary_output': ys_train},
    validation_data = (
        X_test,
        {'role_output': yr_test, 'salary_output': ys_test}
    ),
    epochs     = 60,
    batch_size = 32,
    callbacks  = [
        keras.callbacks.EarlyStopping(
            monitor              = 'val_loss',
            patience             = 10,
            restore_best_weights = True,
            verbose              = 1,
            mode                 = 'min'
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor  = 'val_loss',
            factor   = 0.5,
            patience = 5,
            verbose  = 1,
            min_lr   = 1e-6
        )
    ],
    verbose = 1
)

  TRAINING — MULTITASK TRANSFORMER
Epoch 1/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 3.3981 - role_output_accuracy: 0.2190 - role_output_loss: 2.2083 - salary_output_loss: 1.1564 - salary_output_mae: 0.8057 - val_loss: 2.8120 - val_role_output_accuracy: 0.1514 - val_role_output_loss: 1.6044 - val_salary_output_loss: 1.2318 - val_salary_output_mae: 0.7339 - learning_rate: 3.0000e-04
Epoch 2/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2.4433 - role_output_accuracy: 0.4002 - role_output_loss: 1.6322 - salary_output_loss: 0.8187 - salary_output_mae: 0.6632 - val_loss: 2.7977 - val_role_output_accuracy: 0.1147 - val_role_output_loss: 1.5941 - val_salary_output_loss: 1.2283 - val_salary_output_mae: 0.7448 - learning_rate: 3.0000e-04
Epoch 3/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.9115 - role_output_accuracy: 0.5252 - role_output_loss: 1.2687 - salary_output_loss: 0.6498 - salary_output_mae: 0.6106 - val_loss: 2.7743 - val_role_output_accuracy: 0.1147 - val_role_out

In [ ]:
role_probs, sal_preds_norm = model.predict(X_test, verbose=0)

role_preds = np.argmax(role_probs, axis=1)
accuracy   = accuracy_score(yr_test, role_preds) * 100
sal_preds_eur   = sal_preds_norm.flatten() * sal_std + sal_mean
sal_actuals_eur = ys_test * sal_std + sal_mean
mae             = mean_absolute_error(sal_actuals_eur, sal_preds_eur)

print(f"  EVALUATION RESULTS")

print(f"\n Role Accuracy : {accuracy:.1f}%")
print(f" Salary MAE   : €{mae:,.0f}")

print(f"\n  Per-role classification report:")
print(classification_report(yr_test, role_preds,
                             target_names=le_role.classes_))

  EVALUATION RESULTS

 Role Accuracy : 80.3%
 Salary MAE   : €8,224

  Per-role classification report:
                                              precision    recall  f1-score   support

                           Clinical Research       0.74      0.76      0.75        33
                  Manufacturing & Operations       0.79      0.65      0.71        34
                    Pharmaceutical Marketing       0.73      0.64      0.68        25
Pharmaceutical, Healthcare And Medical Sales       0.89      0.95      0.92       102
                          Regulatory Affairs       0.60      0.62      0.61        24

                                    accuracy                           0.80       218
                                   macro avg       0.75      0.72      0.73       218
                                weighted avg       0.80      0.80      0.80       218



In [ ]:
df_results = pd.DataFrame({
    'role_actual'     : le_role.inverse_transform(yr_test),
    'role_predicted'  : le_role.inverse_transform(role_preds),
    'salary_actual'   : sal_actuals_eur,
    'salary_predicted': sal_preds_eur
})

summary = df_results.groupby('role_actual').agg(
    count        = ('salary_actual', 'count'),
    avg_actual   = ('salary_actual', 'mean'),
    avg_pred     = ('salary_predicted', 'mean'),
    mae_per_role = ('salary_actual',
        lambda x: mean_absolute_error(
            x, df_results.loc[x.index, 'salary_predicted']))
).reset_index()

summary.columns            = ['Role','Count','Avg Actual €',
                               'Avg Predicted €','MAE €']
summary['Avg Actual €']    = summary['Avg Actual €'].round(0).astype(int)
summary['Avg Predicted €'] = summary['Avg Predicted €'].round(0).astype(int)
summary['MAE €']           = summary['MAE €'].round(0).astype(int)

print(f"  ROLE → SALARY MAPPING")
print(f"\n  {'Role':<45} {'N':>5} {'Actual €':>10} "
      f"{'Pred €':>10} {'MAE €':>8}")

for _, row in summary.iterrows():
    print(f"  {row['Role']:<45} {row['Count']:>5} "
          f"{row['Avg Actual €']:>10,} "
          f"{row['Avg Predicted €']:>10,} "
          f"{row['MAE €']:>8,}")

  ROLE → SALARY MAPPING

  Role                                              N   Actual €     Pred €    MAE €
  Clinical Research                                33     68,470     61,066   11,735
  Manufacturing & Operations                       34     59,838     62,350    9,781
  Pharmaceutical Marketing                         25     45,680     43,919    5,924
  Pharmaceutical, Healthcare And Medical Sales    102     45,716     45,658    5,364
  Regulatory Affairs                               24     65,500     53,505   15,745


In [ ]:
print(f"  TRAINING HISTORY (last 5 epochs)")

print(f"\n  {'Epoch':>6} {'Loss':>10} {'Train Acc':>10} "
      f"{'Val Loss':>10} {'Val Acc':>10}")


total_ep = len(history.history['loss'])
for i in range(max(0, total_ep - 5), total_ep):
    print(f"  {i+1:>6} "
          f"{history.history['loss'][i]:>10.4f} "
          f"{history.history['role_output_accuracy'][i]*100:>9.1f}% "
          f"{history.history['val_loss'][i]:>10.4f} "
          f"{history.history['val_role_output_accuracy'][i]*100:>9.1f}%")

  TRAINING HISTORY (last 5 epochs)

   Epoch       Loss  Train Acc   Val Loss    Val Acc
      56     0.4866      87.8%     1.2010      80.3%
      57     0.5178      88.4%     1.2110      79.8%
      58     0.4858      88.2%     1.2094      79.8%
      59     0.4873      89.0%     1.2073      80.3%
      60     0.4742      88.5%     1.1986      78.9%


In [ ]:
clinical     = summary[summary['Role'].str.contains(
    'Clinical', case=False, na=False)]
clin_sal     = int(clinical['Avg Predicted €'].values[0]) \
               if len(clinical) > 0 else int(sal_mean)
clin_mae     = int(clinical['MAE €'].values[0]) \
               if len(clinical) > 0 else int(mae)
best_row     = summary.sort_values('MAE €').iloc[0]

TL="\u2554"; TR="\u2557"; BL="\u255A"; BR="\u255D"
LT="\u2560"; RT="\u2563"; H="\u2550";  V="\u2551"; W=61

def row(text=""): return V + text.ljust(W) + V

print(TL + H*W + TR)
print(row())
print(row("   \U0001f52c  KEY RESEARCH FINDING — RQ1"))
print(row("   Role-Salary Patterns — European Pharma Market"))
print(row())
print(LT + H*W + RT)
print(row())
print(row("   Clinical Research roles predict"))
print(row())
print(row(f"       \u2605  Avg Predicted Salary  \u2192  \u20ac{clin_sal:,}"))
print(row(f"       \u2605  Role MAE              \u2192  \u20ac{clin_mae:,}"))
print(row(f"       \u2605  Overall Accuracy      \u2192  {accuracy:.1f}%"))
print(row(f"       \u2605  Overall Salary MAE    \u2192  \u20ac{int(mae):,}"))
print(row())
print(LT + H*W + RT)
print(row())
print(row(f"   Best per-role MAE:"))
print(row(f"       {best_row['Role']} \u2192 \u20ac{best_row['MAE €']:,}"))
print(row())
print(LT + H*W + RT)
print(row())
print(row("   Gap addressed:"))
print(row("   Ather (2024) + Agrawal (2025) — US/UK tech only"))
print(row("   \u2192 First EU pharma multi-task transformer"))
print(row())
print(BL + H*W + BR)

╔═════════════════════════════════════════════════════════════╗
║                                                             ║
║   🔬  KEY RESEARCH FINDING — RQ1                             ║
║   Role-Salary Patterns — European Pharma Market             ║
║                                                             ║
╠═════════════════════════════════════════════════════════════╣
║                                                             ║
║   Clinical Research roles predict                           ║
║                                                             ║
║       ★  Avg Predicted Salary  →  €61,066                   ║
║       ★  Role MAE              →  €11,735                   ║
║       ★  Overall Accuracy      →  80.3%                     ║
║       ★  Overall Salary MAE    →  €8,224                    ║
║                                                             ║
╠═════════════════════════════════════════════════════════════╣
║                                       